# Unlimited-OCR Transformers Debug

Use this notebook from VS Code Remote with kernel `Python (uocr-debug)`. Load the model once, then rerun later cells while debugging prompts, images, preprocessing, and generation parameters.

In [ ]:
import os
import time
from pathlib import Path

ROOT = Path('/mnt1/yixuan/unlimited-ocr-posttrain')
MODEL_DIR = ROOT / 'models' / 'baidu_Unlimited-OCR'
DEBUG_DIR = ROOT / 'debug'
IMAGE_PATH = DEBUG_DIR / 'test_images' / 'sample_receipt.png'
OUTPUT_DIR = DEBUG_DIR / 'outputs' / 'notebook_single_image'

os.environ.setdefault('HF_ENDPOINT', 'https://hf-mirror.com')
os.environ.setdefault('HF_HOME', str(ROOT / 'hf_cache'))
os.environ.setdefault('TRANSFORMERS_CACHE', str(ROOT / 'hf_cache'))
os.environ.setdefault('HF_HUB_DISABLE_XET', '1')
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('MODEL_DIR =', MODEL_DIR)
print('IMAGE_PATH =', IMAGE_PATH)
print('OUTPUT_DIR =', OUTPUT_DIR)

In [ ]:
import torch
import transformers
from PIL import Image, ImageDraw, ImageFont

print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.version.cuda)
print('device_count', torch.cuda.device_count())
if torch.cuda.is_available():
    print('gpu0', torch.cuda.get_device_name(0))
print('transformers', transformers.__version__)

In [ ]:
def make_sample_image(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    image = Image.new('RGB', (960, 640), 'white')
    draw = ImageDraw.Draw(image)
    try:
        title_font = ImageFont.truetype('DejaVuSans-Bold.ttf', 40)
        body_font = ImageFont.truetype('DejaVuSansMono.ttf', 28)
    except OSError:
        title_font = ImageFont.load_default()
        body_font = ImageFont.load_default()

    lines = [
        ('UNLIMITED OCR DEBUG', title_font),
        ('Invoice No: UOCR-2026-0706', body_font),
        ('Date: 2026-07-06', body_font),
        ('Item                Qty    Price', body_font),
        ('Notebook paper       2     12.50', body_font),
        ('Black pen            5      8.75', body_font),
        ('Total                     21.25', body_font),
    ]
    y = 60
    for text, font in lines:
        draw.text((70, y), text, fill='black', font=font)
        y += 64
    image.save(path)

if not IMAGE_PATH.exists():
    make_sample_image(IMAGE_PATH)

Image.open(IMAGE_PATH)

## Load model once

Run this cell once. After the model is in GPU memory, rerun the later inference/debug cells without restarting the kernel.

In [ ]:
from transformers import AutoModel, AutoTokenizer

t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), trust_remote_code=True)
print(f'tokenizer loaded in {time.time() - t0:.1f}s')

t1 = time.time()
model = AutoModel.from_pretrained(
    str(MODEL_DIR),
    trust_remote_code=True,
    use_safetensors=True,
    torch_dtype=torch.bfloat16,
)
model = model.eval().cuda()
print(f'model loaded in {time.time() - t1:.1f}s')

In [ ]:
# Choose one mode.
# gundam: official single-image crop mode; base: no crop, 1024 image tokens.
mode = 'gundam'
if mode == 'gundam':
    base_size, image_size, crop_mode = 1024, 640, True
else:
    base_size, image_size, crop_mode = 1024, 1024, False

prompt = '<image>document parsing.'
max_length = 32768
no_repeat_ngram_size = 35
ngram_window = 128
temperature = 0.0

print(mode, base_size, image_size, crop_mode)

In [ ]:
t0 = time.time()
result = model.infer(
    tokenizer,
    prompt=prompt,
    image_file=str(IMAGE_PATH),
    output_path=str(OUTPUT_DIR),
    base_size=base_size,
    image_size=image_size,
    crop_mode=crop_mode,
    max_length=max_length,
    no_repeat_ngram_size=no_repeat_ngram_size,
    ngram_window=ngram_window,
    temperature=temperature,
    save_results=True,
)
print('elapsed', round(time.time() - t0, 1), 'seconds')
print('result type:', type(result))

In [ ]:
result_md = OUTPUT_DIR / 'result.md'
print(result_md)
print(result_md.read_text(encoding='utf-8') if result_md.exists() else 'missing result.md')

boxed = OUTPUT_DIR / 'result_with_boxes.jpg'
Image.open(boxed) if boxed.exists() else boxed

## Useful debug handles

Use these cells when stepping into the remote code or checking model internals.

In [ ]:
print(type(model))
print(model.config.model_type)
print(model.config.architectures)
print('tokenizer size:', len(tokenizer))
print('image token id:', tokenizer.convert_tokens_to_ids('<image>'))

In [ ]:
import inspect

print(inspect.signature(model.infer))
print(inspect.getsource(model.infer).splitlines()[0:40])

In [ ]:
# Free GPU memory if needed.
# del model
# torch.cuda.empty_cache()